# 01. Document Intelligence — Extracting Invoice Layout with Azure AI Document Intelligence

This notebook is the first step of the section's CloudXeus invoice-processing pipeline, which combines **Azure AI Document Intelligence** with **Azure AI Foundry agents** wired to an MCP (Model Context Protocol) knowledge base. Here we call the Document Intelligence `prebuilt-layout` model against a sample CloudXeus invoice PDF stored in Azure Blob Storage, and print the extracted content as Markdown — tables, headings and reading order preserved. This is the "raw extraction" building block that later notebooks in this section (`04_invoke_agent.ipynb`, `05_content_safety.ipynb`) build on or parallel with a different extraction service.

**Difficulty: Beginner**


## Prerequisites

**Python packages**
```bash
pip install azure-ai-documentintelligence azure-core python-dotenv
```
`azure-ai-documentintelligence` is not currently pinned in the repo root `requirements.txt`, so install it into the shared `venv` before running this notebook.

**Azure resources**
- An **Azure AI Document Intelligence** resource (the successor to "Form Recognizer"). You need its endpoint and either an API key or an Entra ID identity with the `Cognitive Services User` role.

**Environment variables** (add these to the root `.env`, alongside the existing `AZURE_AI_PROJECT_ENDPOINT` / `AZURE_AI_MODEL_DEPLOYMENT` pair used in `11_azure_ai_foundry/`):

```
AZURE_DOCUMENTINTELLIGENCE_ENDPOINT=https://<your-resource>.cognitiveservices.azure.com/
AZURE_DOCUMENTINTELLIGENCE_KEY=<your-key>
DOCUMENT_URL=https://<your-storage-account>.blob.core.windows.net/.../CloudXeus_Invoice_INV-CX-2026-1002.pdf
```

The original script points `DOCUMENT_URL` at a blob-storage copy of `CloudXeus_Invoice_INV-CX-2026-1002.pdf`. A local copy of the same file (and three siblings, `..._1001.pdf` through `..._1004.pdf`) ships in the sibling `cloudxeus_invoices/` folder in this section — useful if you want to adapt the call to analyze document bytes instead of a URL (see **Try It Yourself** below).

## What You'll Learn

- How to instantiate a `DocumentIntelligenceClient` against an Azure AI Document Intelligence resource
- The long-running-operation ("poller") pattern used by `begin_analyze_document`
- The difference between analyzing a remote URL (`url_source`) vs. local document bytes
- What the `prebuilt-layout` model returns, and why `output_content_format="markdown"` is useful for downstream LLM consumption
- Why this pipeline picks `prebuilt-layout` here rather than a semantic model like `prebuilt-invoice`

### Imports and environment-driven configuration

We load credentials and the target document URL from environment variables via `python-dotenv` instead of hardcoding them, so the notebook can be safely shared without leaking secrets. The original script had `key = ""` and a hardcoded endpoint/URL in plain text — we replace both with `os.getenv(...)` calls.

In [ ]:
# pip install azure-ai-documentintelligence python-dotenv
import os

from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv()

endpoint = os.getenv("AZURE_DOCUMENTINTELLIGENCE_ENDPOINT")
key = os.getenv("AZURE_DOCUMENTINTELLIGENCE_KEY")

# Sample CloudXeus invoice PDF (blob-storage URL). A local copy of the same
# file lives in ../cloudxeus_invoices/CloudXeus_Invoice_INV-CX-2026-1002.pdf
document_url = os.getenv(
    "DOCUMENT_URL",
    "https://stcxai103capdeus01.blob.core.windows.net/course-products/student-invoices/CloudXeus_Invoice_INV-CX-2026-1002.pdf",
)


### Creating the client

`DocumentIntelligenceClient` is the SDK's entry point for both prebuilt and custom document models. We authenticate with an `AzureKeyCredential` here (simplest for a personal resource); pinning `api_version` guards the notebook against breaking changes if Microsoft ships a newer default API version later.

💡 **Exam tip (AI-102):** Document Intelligence resources can also authenticate via Entra ID (`DefaultAzureCredential`) instead of a key — the exam expects you to know both patterns, and to prefer managed identity / Entra ID in production.

🔄 **Alternatives:** You could call the Document Intelligence REST API directly with `requests` instead of the SDK — useful in restricted environments where you can't install the SDK, but you lose the poller convenience and typed response models.

In [ ]:
client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
    api_version="2024-11-30",
)


### Analyzing the document

`begin_analyze_document` is a **long-running operation (LRO)**: it kicks off analysis server-side and immediately returns a *poller* rather than blocking. Calling `poller.result()` blocks until the job finishes and returns the final `AnalyzeResult`. We ask for `output_content_format="markdown"`, which gives us a single `result.content` string with tables rendered as Markdown tables and reading order preserved — convenient for handing straight to an LLM prompt.

💡 **Exam tip (AI-102):** Document Intelligence ships several prebuilt models with different jobs: `prebuilt-read` (OCR only), `prebuilt-layout` (structure — text, tables, selection marks, but no semantic fields), and `prebuilt-invoice` (layout **plus** semantic key-value fields like `InvoiceTotal`, `VendorName`, `DueDate`, each with a confidence score). This script deliberately uses `prebuilt-layout`, not `prebuilt-invoice` — later notebooks in this section (`04_invoke_agent.ipynb`) instead use a **custom Azure AI Content Understanding analyzer** for schema-driven field extraction, so this step's job is just to get clean, LLM-friendly raw text/markdown, not structured fields.

🔄 **Alternatives:** For very large batches, `begin_analyze_document` also has an async counterpart (`azure.ai.documentintelligence.aio.DocumentIntelligenceClient`) so you can `await` many analyses concurrently instead of polling one at a time synchronously.

In [ ]:
poller = client.begin_analyze_document(
    model_id="prebuilt-layout",
    body=AnalyzeDocumentRequest(url_source=document_url),
    output_content_format="markdown",
)

result = poller.result()

print(result.content)


## Summary

We authenticated against an Azure AI Document Intelligence resource, submitted a CloudXeus invoice PDF (by URL) to the `prebuilt-layout` model as an asynchronous long-running job, waited for the result via `poller.result()`, and printed the extracted content as a single Markdown string. `result.content` contains the whole document's text with tables converted to Markdown table syntax — this is the raw material that a later step in this section (`05_content_safety.ipynb`) will run through a content-safety check before ever reaching an LLM prompt.

## Try It Yourself

1. **Easy:** Point `document_url` at one of the other three sample invoices in `../cloudxeus_invoices/` (e.g. `CloudXeus_Invoice_INV-CX-2026-1001.pdf`) and compare how the Markdown table output differs.
2. **Intermediate:** Change `model_id` to `"prebuilt-invoice"` and, instead of printing `result.content`, iterate `result.documents[0].fields` to print each semantic field's value and `confidence` score. Compare which is easier to consume programmatically: raw Markdown or structured fields.
3. **Advanced:** Rewrite the call to analyze a **local file** instead of a blob URL — open one of the sample PDFs in `../cloudxeus_invoices/` with `open(path, "rb")` and pass the bytes via `body=` directly (no `AnalyzeDocumentRequest(url_source=...)`), so the notebook works entirely offline against local sample data.